# Augmented Conversation Evaluation

### Setup & imports

In [ ]:
# Setup & imports
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# DialogRPT-human-vs-rand: Discriminates real vs. random replies.
# DialogRPT-human-vs-machine: Discriminates human vs. machine replies.

# Load the model and tokenizer
model = "microsoft/DialogRPT-human-vs-rand"
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForSequenceClassification.from_pretrained(model)


In [ ]:
#Image description insertion
import os
from pathlib import Path
import re


gen = os.walk(f".\\conversations")
next(gen)

conversations = []

for x in gen:
    directory = Path(x[0])
    text_files = list(directory.rglob('*.txt'))
    for file in text_files:
        with open(file, encoding="utf-8") as file:
            lines = [re.sub(r'^[^:]+:\s*', '', line.rstrip()) for line in file]
            conversations.append(lines[1:])


In [ ]:
# Modeling & evaluation
def score_response(context, response, model, tokenizer):
    inputs = tokenizer.encode_plus(context, response, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        score = model(**inputs)[0].item()
    return score

In [ ]:
# Modeling & evaluation
responses = []

for conv in conversations:
    for i in range(1, len(conv)):
        context = " ".join(conv[:i])
        response = conv[i]
        score = score_response(context, response, model, tokenizer)
        responses.append(response, score)